In [ ]:
# Cell 1: imports, paths, device, transforms

import multiprocessing
from pathlib import Path

import torch
torch.backends.cudnn.benchmark = True

from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets

# Try to use transforms.v2 (faster C++ backend) and fallback to classic transforms
try:
    from torchvision.transforms import v2 as T
    HAVE_V2 = True
    print("Using torchvision.transforms.v2")
except ImportError:
    from torchvision import transforms as T
    HAVE_V2 = False
    print("Using torchvision.transforms (legacy)")

# Paths
REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
SW_PATCH_ROOT = REPO_ROOT / "data" / "patches_sw"

print("Repo root:", REPO_ROOT)
print("Sliding-window patch root:", SW_PATCH_ROOT)

# GPU and multiprocessing setup 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if device.type == "cuda":
    print("CUDA available:", torch.cuda.is_available())
    print("GPU name:", torch.cuda.get_device_name(0))

multiprocessing.set_start_method("spawn", force=True)
print("Multiprocessing start method set to 'spawn'")

# Hyperparameters
IMG_SIZE = 160
BATCH_SIZE = 128
NUM_WORKERS = 4
PREFETCH_FACTOR = 2
PIN_MEMORY = (device.type == "cuda")

EPOCHS = 10
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

# mean/std for PCB-like boards (approx)
MEAN = (0.5, 0.5, 0.5)
STD  = (0.25, 0.25, 0.25)

if HAVE_V2:
    train_transform = T.Compose([
        T.ToImage(),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=5),
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(MEAN, STD),
    ])

    eval_transform = T.Compose([
        T.ToImage(),
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(MEAN, STD),
    ])
else:
    train_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(degrees=5),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])

    eval_transform = T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


In [ ]:
# Cell 2: datasets and dataloaders

from torchvision import datasets

train_dir = SW_PATCH_ROOT / "train"
val_dir   = SW_PATCH_ROOT / "valid"
test_dir  = SW_PATCH_ROOT / "test"

train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset   = datasets.ImageFolder(val_dir,   transform=eval_transform)
test_dataset  = datasets.ImageFolder(test_dir,  transform=eval_transform)

classes = train_dataset.classes
num_classes = len(classes)

print("Classes:", classes)
print("Train size:", len(train_dataset))
print("Valid size:", len(val_dataset))
print("Test  size:", len(test_dataset))

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=True,
    prefetch_factor=PREFETCH_FACTOR,
)

print("Batches -> train:", len(train_loader),
      "valid:", len(val_loader), "test:", len(test_loader))


In [ ]:
# Cell 3: model, loss, optimizer, scheduler, scaler

from torchvision import models
from torch.cuda.amp import GradScaler

model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)

in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

scaler = GradScaler(enabled=(device.type == "cuda"))

print("Model ready on", device)


In [ ]:
# Cell 4: train / eval epoch functions

from tqdm.auto import tqdm
from torch.amp import autocast  

def train_one_epoch(model, dataloader, optimizer, criterion, scaler, device,
                    epoch=None, total_epochs=None):
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{total_epochs} [Train]", leave=False)

    for inputs, targets in pbar:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with autocast(device_type=device.type, dtype=torch.float16,
                      enabled=(device.type == "cuda")):
            outputs = model(inputs)
            loss = criterion(outputs, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        batch_size = inputs.size(0)
        running_loss += loss.item() * batch_size
        _, preds = outputs.max(1)
        running_correct += preds.eq(targets).sum().item()
        total += batch_size

        pbar.set_postfix({
            "loss": f"{running_loss/total:.4f}",
            "acc": f"{running_correct/total:.4f}"
        })

    return running_loss / total, running_correct / total


def eval_one_epoch(model, dataloader, criterion, device,
                   epoch=None, total_epochs=None):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch}/{total_epochs} [Eval]", leave=False)

    with torch.no_grad():
        for inputs, targets in pbar:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            with autocast(device_type=device.type, dtype=torch.float16,
                          enabled=(device.type == "cuda")):
                outputs = model(inputs)
                loss = criterion(outputs, targets)

            batch_size = inputs.size(0)
            running_loss += loss.item() * batch_size
            _, preds = outputs.max(1)
            running_correct += preds.eq(targets).sum().item()
            total += batch_size

            pbar.set_postfix({
                "loss": f"{running_loss/total:.4f}",
                "acc": f"{running_correct/total:.4f}"
            })

    return running_loss / total, running_correct / total


In [ ]:
# Cell 5: main training loop (retraining, new model filename)

RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True, parents=True)

best_path = RESULTS_DIR / "mobilenetv2_sw_retrained_best.pth"

best_val_acc = 0.0

print(f"Starting training for {EPOCHS} epochs on", device)
for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device,
        epoch=epoch, total_epochs=EPOCHS
    )

    val_loss, val_acc = eval_one_epoch(
        model, val_loader, criterion, device,
        epoch=epoch, total_epochs=EPOCHS
    )

    scheduler.step(val_acc)

    print(f"Train loss: {train_loss:.4f}, Train acc: {train_acc:.4f}")
    print(f"Val   loss: {val_loss:.4f}, Val   acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), best_path)
        print("New best model saved!")

print("\nTraining done.")
print("Best validation accuracy:", best_val_acc)
print("Best model path:", best_path)


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch

model.eval()

# pick a random test sample
idx = random.randrange(len(test_dataset))
img_tensor, label_idx = test_dataset[idx]  
true_label = classes[label_idx]

# run through model
with torch.no_grad(), torch.amp.autocast(
    device_type=device.type,
    dtype=torch.float16,
    enabled=(device.type == "cuda")
):
    logits = model(img_tensor.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=1)[0].cpu().numpy()

pred_idx = int(np.argmax(probs))
pred_label = classes[pred_idx]

# unnormalize for display
img_np = img_tensor.permute(1, 2, 0).cpu().numpy()
mean = np.array(MEAN)
std = np.array(STD)
img_vis = np.clip(img_np * std + mean, 0, 1)

plt.figure(figsize=(3,3))
plt.imshow(img_vis)
plt.axis("off")
plt.title(f"True: {true_label} | Pred: {pred_label}")
plt.show()

print("Top class probabilities:")
for cls, p in sorted(zip(classes, probs), key=lambda x: x[1], reverse=True):
    print(f"{cls:15s}: {p:0.3f}")


In [ ]:
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# make a test loader (safe even if one already exists)
test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=0,                
    pin_memory=(device.type == "cuda"),
)

model.eval()
all_labels = []
all_preds = []
correct = 0
total = 0

with torch.no_grad():
    for inputs, targets in test_loader:
        inputs = inputs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.amp.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            outputs = model(inputs)

        preds = outputs.argmax(dim=1)
        correct += (preds == targets).sum().item()
        total += targets.size(0)

        all_labels.append(targets.cpu())
        all_preds.append(preds.cpu())

test_acc = correct / total
all_labels = torch.cat(all_labels).numpy()
all_preds = torch.cat(all_preds).numpy()

print(f"\nPatch-level TEST accuracy: {test_acc:.4f}\n")

print(classification_report(all_labels, all_preds, target_names=classes, digits=3))

cm = confusion_matrix(all_labels, all_preds, labels=np.arange(len(classes)))
cm_df = pd.DataFrame(cm, index=classes, columns=classes)
print("\nConfusion matrix:\n")
display(cm_df)
